In [62]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler , OrdinalEncoder

df = pd.read_csv("../data/raw/student_mental_health.csv")

# Étape 2 — Cleaning & Preprocessing

# 1 - Train/Test Split

In [49]:
X = df.drop(columns=["Student_ID", "Mental_Health_Risk"])
y = df["Mental_Health_Risk"]


X_train , X_test , y_train , y_test = train_test_split(
    X,
    y,
    test_size = 0.20,
    random_state = 42,
    stratify = y,
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (1600, 15)
X_test: (400, 15)
y_train: (1600,)
y_test: (400,)


In [50]:
numerical_columns = [
    "Age",
    "Year_of_Study",
    "Sleep_Hours",
    "Study_Hours_Per_Week",
    "GPA",
    "Social_Media_Hours",
    "Screen_Time_Hours",
    "Financial_Stress",
    "Family_Support",
    "Attendance_Rate"
]

nominal_columns = [
    "Gender",
    "Part_Time_Job",
    "Previous_Mental_Health_History"
]

ordinal_columns = [
    "Physical_Activity",
    "Diet_Quality"
]

# 2 - Inconsistencies Treatment

In [51]:
invalid_train = (
    X_train["Social_Media_Hours"] > X_train["Screen_Time_Hours"]
)

invalid_test = (
    X_test["Social_Media_Hours"] > X_test["Screen_Time_Hours"]
)

print("Training inconsistencies:", invalid_train.sum())
print("Testing inconsistencies:", invalid_test.sum())

X_train.loc[invalid_train, "Social_Media_Hours"] = np.nan
X_test.loc[invalid_test, "Social_Media_Hours"] = np.nan

Training inconsistencies: 27
Testing inconsistencies: 10


# 3 - Vérification des valeurs manquantes

In [52]:
print("Missing values in Training Set:")
print(X_train.isnull().sum())

print("\nMissing values in Testing Set:")
print(X_test.isnull().sum())

Missing values in Training Set:
Age                                0
Gender                             0
Year_of_Study                      0
Sleep_Hours                       54
Study_Hours_Per_Week               0
GPA                               36
Social_Media_Hours                75
Screen_Time_Hours                  0
Physical_Activity                  0
Financial_Stress                  76
Family_Support                    36
Attendance_Rate                   39
Part_Time_Job                      0
Diet_Quality                      55
Previous_Mental_Health_History     0
dtype: int64

Missing values in Testing Set:
Age                                0
Gender                             0
Year_of_Study                      0
Sleep_Hours                        8
Study_Hours_Per_Week               0
GPA                                2
Social_Media_Hours                20
Screen_Time_Hours                  0
Physical_Activity                  0
Financial_Stress                  1

# 4 - Détection des valeurs aberrantes

In [53]:
Q1 = X_train[numerical_columns].quantile(0.25)
Q3 = X_train[numerical_columns].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_train = (
    (X_train[numerical_columns] < lower_bound) |
    (X_train[numerical_columns] > upper_bound)
)

outliers_test = (
    (X_test[numerical_columns] < lower_bound) |
    (X_test[numerical_columns] > upper_bound)
)

print("Outliers in Training Set:")
print(outliers_train.sum())

print("\nOutliers in Testing Set:")
print(outliers_test.sum())

Outliers in Training Set:
Age                      2
Year_of_Study            0
Sleep_Hours              9
Study_Hours_Per_Week     7
GPA                      5
Social_Media_Hours      10
Screen_Time_Hours        7
Financial_Stress         0
Family_Support           0
Attendance_Rate         10
dtype: int64

Outliers in Testing Set:
Age                     2
Year_of_Study           0
Sleep_Hours             3
Study_Hours_Per_Week    0
GPA                     1
Social_Media_Hours      6
Screen_Time_Hours       0
Financial_Stress        0
Family_Support          0
Attendance_Rate         1
dtype: int64


# 5 - Traitement des valeurs aberrantes

In [54]:
X_train[numerical_columns] = X_train[numerical_columns].clip(
    lower=lower_bound,
    upper=upper_bound,
    axis=1,
)

X_test[numerical_columns] = X_test[numerical_columns].clip(
    lower=lower_bound,
    upper=upper_bound,
    axis=1,
)

# 6 - Création d'un pipeline pour les caractéristiques numériques

In [55]:
# skewness = X_train[numerical_columns].skew()
# print(skewness.sort_values())

numerical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="mean")),
        ("sclaer", StandardScaler()),
    ]
)

# 7 - Création d'un pipeline pour les caractéristiques nominales

In [56]:
nominal_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("scaler", OneHotEncoder(drop="first" , handle_unknown="ignore")),
    ]
)

# 8 - Création d'un pipeline pour les caractéristiques ordinales

In [57]:
ordinal_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "scaler",
            OrdinalEncoder(
                categories=[["Low", "Moderate", "High"], ["Poor", "Average", "Good"]],
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
        ),
    ]
)

# 9 - Création d'un ColumnTransformer

In [58]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_columns),
        ("nom", nominal_pipeline, nominal_columns),
        ("ord", ordinal_pipeline, ordinal_columns),
    ]
)

# 10 - Application du prétraitement aux données d'entraînement et de test

In [59]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)

print("\nMissing values in training:", np.isnan(X_train_processed).sum())

print("Missing values in testing:", np.isnan(X_test_processed).sum())

Training shape: (1600, 16)
Testing shape: (400, 16)

Missing values in training: 0
Missing values in testing: 0


In [60]:
print("Training Target Distribution:")
print(y_train.value_counts())

print("\nTesting Target Distribution:")
print(y_test.value_counts())

Training Target Distribution:
Mental_Health_Risk
High      544
Medium    528
Low       528
Name: count, dtype: int64

Testing Target Distribution:
Mental_Health_Risk
High      136
Medium    132
Low       132
Name: count, dtype: int64


In [61]:
print("Training percentages:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting percentages:")
print(y_test.value_counts(normalize=True) * 100)

Training percentages:
Mental_Health_Risk
High      34.0
Medium    33.0
Low       33.0
Name: proportion, dtype: float64

Testing percentages:
Mental_Health_Risk
High      34.0
Medium    33.0
Low       33.0
Name: proportion, dtype: float64


In [63]:
Path("../data/processed").mkdir(parents=True, exist_ok=True)

joblib.dump(
    {
        "X_train": X_train_processed,
        "X_test": X_test_processed,
        "y_train": y_train,
        "y_test": y_test,
        "preprocessor": preprocessor,
    },"../data/processed/preprocessed_data.joblib")

['../data/processed/preprocessed_data.joblib']